# KTIR Latency Estimation Demo

This notebook shows how to use `ktir_cpu` to get cycle-approximate latency estimates
for KTIR kernels — no Spyre hardware required.

**What we cover:**
1. [Matmul — load, run, report](#1.-Matmul-—-load,-run,-report)
2. [Per-core breakdown](#2.-Per-core-breakdown)
3. [Softmax — compare with matmul](#3.-Softmax-—-compare-with-matmul)
4. [Multi-kernel roofline](#4.-Multi-kernel-roofline)
5. [Per-op trace: paged attention](#5.-Per-op-trace:-paged-attention)
6. [Chip-level analysis](#6.-Chip-level-analysis-—-the-whole-chip-as-one-unit)
7. [Tuning HardwareConfig](#7.-Tuning-HardwareConfig)
8. [MLIR frontend parser (optional)](#8.-MLIR-frontend-parser-(optional))
9. [Cross-core communication (comm) view](#9.-Cross-core-communication-(comm)-view)

## Setup

```bash
uv sync --group dev   # installs jupyter + matplotlib
```

Then launch:

```bash
uv run jupyter notebook notebooks/latency_demo.ipynb
```

In [ ]:
import numpy as np
from ktir_cpu.interpreter import KTIRInterpreter
from ktir_cpu.latency import HardwareConfig
from demo_helpers import (
    gen_matmul_mlir, gen_softmax_mlir, gen_sdpa_mlir, gen_paged_attention_mlir,
    run_kernel, run_kernel_matmul, run_kernel_softmax, run_kernel_sdpa, run_kernel_pa,
    show_mlir, plot_roofline,
    print_hw_config, print_core_roofline, print_per_core_table,
    print_kernel_comparison, print_chip_comparison, print_scaling_table, print_trace_summary,
    make_sdpa_tensors, make_pa_tensors, make_pa_scalars,
)

*`plot_roofline`, generators, and print helpers are in [`demo_helpers.py`](demo_helpers.py).*

---

## 1. Matmul — load, run, report

We generate the MLIR inline so the grid, tile sizes, and problem dimensions are all
tunable from the notebook. The grid `[grid_m, grid_n]` determines how many cores run;
each core computes one `BLOCK_SIZE_M × BLOCK_SIZE_N` output tile, accumulating over K
in `K / BLOCK_SIZE_K` steps.

Each core independently loads its own A and B slices (different offsets via `pid_m`,
`pid_n`) — the work is already decomposed across cores at the MLIR level.

In [ ]:
hw = HardwareConfig(lx_size_mb=2, hbm_bandwidth_tb_s=1.024)
print_hw_config(hw)

# --- Matmul parameters (tune these) ---
M, K, N = 256, 256, 256
BLOCK_SIZE_M, BLOCK_SIZE_N, BLOCK_SIZE_K = 128, 128, 128

mlir_text = gen_matmul_mlir(M, N, K, BLOCK_SIZE_M, BLOCK_SIZE_N, BLOCK_SIZE_K)
grid_m, grid_n = M // BLOCK_SIZE_M, N // BLOCK_SIZE_N
print(f"\nMatmul {M}×{K}×{N}, tiles {BLOCK_SIZE_M}×{BLOCK_SIZE_N}×{BLOCK_SIZE_K}, grid [{grid_m},{grid_n}] = {grid_m*grid_n} cores")
# show_mlir(mlir_text)  # uncomment to view generated MLIR

interp = KTIRInterpreter(latency_config=hw)
interp.load(mlir_text)

In [ ]:
a = np.random.rand(M, K).astype(np.float16)
b = np.random.rand(K, N).astype(np.float16)
c = np.zeros((M, N), dtype=np.float16)

outputs = interp.execute_function(
    "matmul_kernel",
    a_ptr=a, b_ptr=b, c_ptr=c,
    K=K, BLOCK_SIZE_M=BLOCK_SIZE_M, BLOCK_SIZE_N=BLOCK_SIZE_N, BLOCK_SIZE_K=BLOCK_SIZE_K,
)

# Verify correctness against NumPy
expected = a @ b
print("Max abs error:", np.max(np.abs(outputs["c_ptr"].astype(np.float32) - expected.astype(np.float32))))

In [ ]:
report = interp.get_latency_report()
print(report)

<style>
table { margin-left: 0 !important; }
td, th { text-align: left !important; }
</style>

### Metrics in the Latency and Roofline Reports

Two methods, one per granularity: **`report.chip_roofline()`** (whole chip as one unit; plain NCU-style names)
and **`report.core_roofline()`** (critical-path core; `core_*` names). `report.roofline()` returns both merged.

**General**

| Field | Meaning |
|:---|:---|
| `kernel_cycles` | Max total cycles across all cores (critical path) |
| `kernel_time_us` | Cycles ÷ clock (default 1 GHz → 1 cycle = 1 ns) |
| `bottleneck` | Which category dominates the critical-path core: `compute`, `memory`, or `comm` |
| Per-core table | `compute_cycles`, `memory_cycles`, `comm_cycles` per core |

**Chip-level** (whole chip as one unit; aggregates over all cores, no `÷ num_cores` per-core inputs)

| Field | Meaning |
|:---|:---|
| `AI` | `Σ FLOP / Σ HBM bytes` over all cores (FLOP/B) — the roofline x-axis |
| `compute_throughput` | `Σ FLOP / elapsed / chip_peak` — compute SOL (fraction of chip compute peak); Nsight *Compute (SM) Throughput %* analogue |
| `dram_throughput` | `Σ HBM bytes / (hbm_bw × elapsed)` — memory SOL (fraction of aggregate HBM bandwidth); `hbm_bw` read directly from `hbm_bandwidth_tb_s` |
| `attainment` | `achieved / ceiling` — how close to the roofline ceiling at the kernel's operating AI |
| `mean_core_active_frac` | `Σ core.total_cycles / (num_cores × elapsed_cycles)` — time-based core occupancy (active = compute + memory + comm); Nsight *SM Active %* analogue |
| `grid_coverage` | `cores_active / num_cores` — **spatial** dispatch coverage (not core busyness; *not* Nsight *SM Active %* — that role is `mean_core_active_frac`) |
| `ridge` | `peak_compute / peak_bw` (FLOP/B) — the AI where memory ceiling meets compute ceiling |

**Per-core** (critical-path core; `core_` prefix)

| Field | Meaning |
|:---|:---|
| `core_AI` | dominant unit FLOPs ÷ HBM bytes on the critical core (FLOP/B) |
| `core_attainment` | `achieved / ceiling` for the dominant unit — per-active-core throughput vs the roofline ceiling at this kernel's AI |
| `core_dominant_unit` | `systolic` or `simd` — the unit with the most compute cycles |
| `ridge` (per-unit) | AI where BW ceiling meets compute ceiling — left = memory-bound, right = compute-bound; differs from chip ridge because per-core BW = `hbm_bw_chip / cores_active` |
| Per-unit `peak_gflops` | per-core flat peak for that compute unit |

In [ ]:
print_core_roofline(report, hw)

---

## 2. Per-core breakdown

In this notebook, serial execution model of compute cores of the accelerator is assumed. The elapsed time of core is split into 3 categories: compute, memroy (access), and (core-to-core) comm (unication). The 'Per-core' her is from the perspective of a single core, when communication is not modeled, the compute-capacity of a core is defined for a given HW configuration, the data-move capacity of a core is a fraction of the total HBM bandwidth that are shared among all active cores (NOT total cores in the accelerator).

In [ ]:
print_per_core_table(report)

---

## 3. Softmax — compare with matmul

In [ ]:
# --- Softmax parameters (tune these) ---
SOFTMAX_N_ROWS = 64
SOFTMAX_ROW_WIDTH = 1024
SOFTMAX_CORES = 4   # match matmul's 4 cores for apples-to-apples comparison

softmax_mlir = gen_softmax_mlir(SOFTMAX_N_ROWS, SOFTMAX_ROW_WIDTH, SOFTMAX_CORES)
print(f"Softmax: {SOFTMAX_N_ROWS} rows × {SOFTMAX_ROW_WIDTH} cols, grid=[{SOFTMAX_CORES},1] = {SOFTMAX_CORES} cores")
print(f"  Each core processes {SOFTMAX_N_ROWS // SOFTMAX_CORES} rows")
# show_mlir(softmax_mlir)  # uncomment to view generated MLIR

softmax_interp = KTIRInterpreter(latency_config=hw)
softmax_interp.load(softmax_mlir)

input_data  = np.random.randn(SOFTMAX_N_ROWS, SOFTMAX_ROW_WIDTH).astype(np.float16)
output_data = np.zeros((SOFTMAX_N_ROWS, SOFTMAX_ROW_WIDTH), dtype=np.float16)

softmax_interp.execute_function(
    "softmax_kernel",
    output_ptr=output_data, input_ptr=input_data,
    n_rows=SOFTMAX_N_ROWS,
)

softmax_report = softmax_interp.get_latency_report()
print(softmax_report)

In [ ]:
print_kernel_comparison([("matmul", report), ("softmax", softmax_report)])

---

## 4. Multi-kernel roofline

Run four kernels through the latency estimator and plot them together:
- **matmul** and **sdpa** — systolic-dominant (matmul ops)
- **softmax** and **paged_attn** — simd-dominant (elementwise / transcendental ops)

**SDPA** (Scaled Dot-Product Attention) computes `softmax(Q @ K^T / √d) @ V`. This is the
core attention primitive in every Transformer — our version is a naive single-pass fused
kernel (no K-tiling). Flash Attention tiles over K blocks and uses online softmax to avoid
materializing the full attention matrix; SDPA here shows what the un-tiled baseline looks like.

**Paged attention** extends SDPA with (1) a KV cache stored in fixed-size pages, (2) an
indirection table (`block_tables`) so pages can be allocated non-contiguously, and (3) online
softmax (the Flash Attention trick) streaming over pages. This is the decode-time attention
pattern in vLLM / TGI / SGLang — the indirect loads via `construct_indirect_access_tile` are
the key KTIR modelling challenge.

In [ ]:
rng = np.random.default_rng(0)

# --- Matmul (4 cores, high K for AI) ---
MM_M, MM_N, MM_K = 256, 256, 128
MM_BM, MM_BN, MM_BK = 128, 128, 128
mm_grid = (MM_M // MM_BM, MM_N // MM_BN)
print(f"Matmul: {MM_M}×{MM_K}×{MM_N}, tiles {MM_BM}×{MM_BN}×{MM_BK}, grid={mm_grid} = {mm_grid[0]*mm_grid[1]} cores")
matmul_report = run_kernel_matmul(hw, MM_M, MM_N, MM_K, MM_BM, MM_BN, MM_BK, rng)

# --- Softmax (small, 4 cores) ---
SM_ROWS, SM_WIDTH, SM_CORES = 64, 1024, 4
print(f"Softmax: {SM_ROWS}×{SM_WIDTH}, grid=[{SM_CORES},1] = {SM_CORES} cores")
softmax_full_report = run_kernel_softmax(hw, SM_ROWS, SM_WIDTH, SM_CORES, rng)

# --- SDPA (4 cores, smaller hd for speed, same AI structure) ---
SDPA_SEQ_LEN, SDPA_HEAD_DIM, SDPA_BLOCK_M = 64, 128, 16
sdpa_grid = SDPA_SEQ_LEN // SDPA_BLOCK_M
print(f"SDPA: seq_len={SDPA_SEQ_LEN}, head_dim={SDPA_HEAD_DIM}, block_m={SDPA_BLOCK_M}, grid=[{sdpa_grid}] = {sdpa_grid} cores")
sdpa_report = run_kernel_sdpa(hw, SDPA_SEQ_LEN, SDPA_HEAD_DIM, SDPA_BLOCK_M, rng)

# --- Paged Attention (4 cores, shorter context, smaller hd) ---
PA_NUM_TOKENS, PA_CONTEXT_LEN = 64, 512
PA_NUM_QUERY_HEADS, PA_NUM_KV_HEADS = 4, 1
PA_HEAD_DIM, PA_BLOCK_SIZE, PA_BLOCK_Q = 64, 64, 16

pa_grid = (PA_NUM_TOKENS // PA_BLOCK_Q, PA_NUM_KV_HEADS)
pa_num_tiles = (PA_CONTEXT_LEN + PA_BLOCK_SIZE - 1) // PA_BLOCK_SIZE
print(f"Paged Attention: {PA_NUM_TOKENS} tokens, ctx={PA_CONTEXT_LEN}, grid={pa_grid} = {pa_grid[0]*pa_grid[1]} cores")
print(f"  num_tiles={pa_num_tiles} iterations")
paged_attn_report = run_kernel_pa(hw, PA_NUM_TOKENS, PA_CONTEXT_LEN, PA_NUM_QUERY_HEADS,
    PA_NUM_KV_HEADS, PA_HEAD_DIM, PA_BLOCK_SIZE, PA_BLOCK_Q, rng)

kernels = [("matmul", matmul_report), ("softmax", softmax_full_report),
           ("sdpa", sdpa_report), ("paged_attn", paged_attn_report)]
print_kernel_comparison(kernels)

In [ ]:
# --- Set 1: 4-core, tiles 256³ ---
M, N, K = 512, 512, 256
BM, BN, BK = 256, 256, 256

mm_1   = run_kernel_matmul(hw, M, N, K, BM, BN, BK, rng)
sm_1   = run_kernel_softmax(hw, SM_ROWS, SM_WIDTH, num_cores=4, rng=rng)
sdpa_1 = run_kernel_sdpa(hw, SDPA_SEQ_LEN, SDPA_HEAD_DIM, SDPA_BLOCK_M, rng)
pa_1   = run_kernel_pa(hw, PA_NUM_TOKENS, PA_CONTEXT_LEN, PA_NUM_QUERY_HEADS,
             PA_NUM_KV_HEADS, PA_HEAD_DIM, PA_BLOCK_SIZE, PA_BLOCK_Q, rng)

# --- Set 2: strong-scaling, 32-core (same problem, smaller tiles) ---
mm_2   = run_kernel_matmul(hw, M, N, K, BM // 4, BN // 2, BK, rng)
sm_2   = run_kernel_softmax(hw, SM_ROWS, SM_WIDTH, num_cores=32, rng=rng)
sdpa_2 = run_kernel_sdpa(hw, SDPA_SEQ_LEN, SDPA_HEAD_DIM, SDPA_BLOCK_M // 8, rng)
pa_2   = run_kernel_pa(hw, PA_NUM_TOKENS, PA_CONTEXT_LEN, PA_NUM_QUERY_HEADS,
             PA_NUM_KV_HEADS, PA_HEAD_DIM, PA_BLOCK_SIZE, PA_BLOCK_Q // 8, rng)

# --- Set 3: weak-scaling, 32-core (8x problem, same tiles) ---
mm_3   = run_kernel_matmul(hw, M * 4, N * 2, K, BM, BN, BK, rng)
sm_3   = run_kernel_softmax(hw, SM_ROWS * 8, SM_WIDTH, num_cores=32, rng=rng)
sdpa_3 = run_kernel_sdpa(hw, SDPA_SEQ_LEN * 8, SDPA_HEAD_DIM, SDPA_BLOCK_M, rng)
pa_3   = run_kernel_pa(hw, PA_NUM_TOKENS * 8, PA_CONTEXT_LEN, PA_NUM_QUERY_HEADS,
             PA_NUM_KV_HEADS, PA_HEAD_DIM, PA_BLOCK_SIZE, PA_BLOCK_Q, rng)

# All 12 experiments: ● set 1 (4-core), ■ set 2 (strong 32), ▲ set 3 (weak 32)
all_kernels = [
    ("matmul (4c)", mm_1, "o"), ("softmax (4c)", sm_1, "o"),
    ("sdpa (4c)", sdpa_1, "o"), ("paged_attn (4c)", pa_1, "o"),
    ("matmul (strong-32)", mm_2, "s"), ("softmax (strong-32)", sm_2, "s"),
    ("sdpa (strong-32)", sdpa_2, "s"), ("paged_attn (strong-32)", pa_2, "s"),
    ("matmul (weak-32)", mm_3, "^"), ("softmax (weak-32)", sm_3, "^"),
    ("sdpa (weak-32)", sdpa_3, "^"), ("paged_attn (weak-32)", pa_3, "^"),
]
print_chip_comparison(all_kernels)

---

## 5. Per-op trace: paged attention

Paged attention uses `construct_indirect_access_tile` to load K and V tiles via
`block_tables` — an indirection table mapping sequence positions to KV-cache pages.
Re-running with `trace_latency=True` and grouping by op type shows where cycles go.

**Scenario:** 64 query tokens attending over 512 context tokens via 8 KV-cache pages
(`block_size=64`). Grid = [4, 1] = 4 cores, GQA ratio = 4 queries per KV head.
Each core processes `block_q=16` query tokens (16 tokens × 4 GQA heads = 64 query rows)
with online softmax streaming through 8 indirect K/V page loads.

In [ ]:
paged_attn_mlir = gen_paged_attention_mlir(PA_NUM_TOKENS, PA_CONTEXT_LEN,
    PA_NUM_QUERY_HEADS, PA_NUM_KV_HEADS, PA_HEAD_DIM, PA_BLOCK_SIZE, PA_BLOCK_Q)

trace_interp = KTIRInterpreter(latency_config=hw, trace_latency=True)
trace_interp.load(paged_attn_mlir)

pa_tensors = make_pa_tensors(PA_NUM_TOKENS, PA_NUM_QUERY_HEADS, PA_NUM_KV_HEADS,
                             PA_HEAD_DIM, PA_BLOCK_SIZE, PA_CONTEXT_LEN, rng)
pa_scalars = make_pa_scalars(PA_CONTEXT_LEN, PA_BLOCK_SIZE, PA_HEAD_DIM)
trace_interp.execute_function("paged_attention_kernel", **pa_tensors, **pa_scalars)

trace_report = trace_interp.get_latency_report()
print_trace_summary(trace_report)

In [ ]:
# Per-core roofline: ● 4-core, ■ strong-32, ▲ weak-32
plot_roofline(hw, all_kernels, granularity="core", title="Per-core roofline — 4c / strong-32 / weak-32")

---

## 6. Chip-level analysis — the whole chip as one unit

Above covers the **per-core** analysis (the critical core vs its own ceiling). What follows is a
**separate analysis at a different granularity**: it collapses the chip to a single unit (the way Nsight
views a device) and asks *how much of the **entire chip** did this kernel use, and what bounds it* — a
question per-core cannot answer.

**Chip-level analysis** is most common scenario for a relative weak AI accelerators to host the kernels of ever growing LLM models, thus complements the **per-core analysis**. A core can have **~99% attainment**
(`core_attainment`) while the **chip is mostly idle**. For instance, chip-level analysis makes visible:

- `grid_coverage` / `mean_core_active_frac` — how many cores actually did work / for how much of the time.
- `dram_throughput` — whether the kernel saturates the **whole chip's** HBM bandwidth (device memory-bound).
- `compute_throughput` — the whole chip's compute utilisation vs its flat peak.

Convention: plain NCU-style names (`AI`, `compute_throughput`, …) are **chip-level**; a `core_` prefix is
**per-core**. The two are never compared on the same axes — their denominators differ.

In [ ]:
print_chip_comparison(kernels)

In [ ]:
# Chip-level roofline: ● 4-core, ■ strong-32, ▲ weak-32
plot_roofline(hw, all_kernels, granularity="chip", title="Chip-level roofline — 4c / strong-32 / weak-32")

**Reading the chip metrics — what per-core hid**

- Kernels that fill only a few cores show low `grid_coverage` / `mean_core_active_frac` and tiny
  `compute_throughput` / `dram_throughput`, *even when* `core_attainment` is ~99%: the busy core is
  efficient, but **most of the chip is idle**. Per-core alone would read as "great".
- Kernels that fill all cores and push `dram_throughput` toward 100% are **HBM-bound**: they
  saturate the whole chip's memory bandwidth — a chip-level statement the per-core view cannot make.

In [ ]:
print("### Scaling experiment results\n")
print_scaling_table(all_kernels)

# Ridge = per-core peak (F/cycle) / per-core BW (B/cycle)
#       = flops_per_cycle / (chip_bw_per_cycle / cores_active)
chip_bw_per_cycle = hw.hbm_bw_chip / hw.clock_hz
print(f"\nRidge (systolic / SIMD) by cores_active:")
for nc in [4, 32]:
    bw = chip_bw_per_cycle / nc
    sys_ridge = hw.systolic_flops_per_cycle / bw
    simd_ridge = hw.simd_elements_per_cycle / bw
    print(f"  {nc} cores: systolic = {sys_ridge:.0f} F/B, SIMD = {simd_ridge:.1f} F/B")

---

## 7. Tuning HardwareConfig

`HardwareConfig` lets you model different hardware targets by adjusting bandwidth, clock, and SIMD width.
All fields default to reasonable approximations — override with real Spyre specs when available.

In [ ]:
# Contrast: same softmax kernel, different HBM bandwidth
hw_lo = HardwareConfig(
    num_cores=32, clock_ghz=1.0, lx_size_mb=2,
    hbm_bandwidth_tb_s=0.256,        # 256 GB/s
    ring_bandwidth_tb_s=0.064,
    simd_elements_per_cycle=64, systolic_rows=8, transcendental_penalty=4,
)
hw_hi = HardwareConfig(
    num_cores=32, clock_ghz=1.0, lx_size_mb=2,
    hbm_bandwidth_tb_s=1.024,        # 1 TB/s
    ring_bandwidth_tb_s=0.064,
    simd_elements_per_cycle=64, systolic_rows=8, transcendental_penalty=4,
)

print("=== Low BW (256 GB/s) ===")
print_hw_config(hw_lo)
print("\n=== High BW (1 TB/s) ===")
print_hw_config(hw_hi)

# Run softmax on both configs (4 cores active)
sm_lo = run_kernel_softmax(hw_lo, SM_ROWS, SM_WIDTH, SM_CORES, rng)
sm_hi = run_kernel_softmax(hw_hi, SM_ROWS, SM_WIDTH, SM_CORES, rng)

print(f"\n{'Config':<12} {'Cycles':>10} {'Bottleneck':>12}  {'SIMD ridge (4c)':>16}")
print(f"{'─'*12} {'─'*10} {'─'*12}  {'─'*16}")
for label, r, h in [("256 GB/s", sm_lo, hw_lo), ("1 TB/s", sm_hi, hw_hi)]:
    bw_per_core = h.hbm_bw_chip / h.clock_hz / SM_CORES
    ridge = h.simd_elements_per_cycle / bw_per_core
    print(f"{label:<12} {r.kernel_cycles:>7.0f} cy {r.bottleneck:>12}  {ridge:>13.1f} F/B")
speedup = sm_lo.kernel_cycles / sm_hi.kernel_cycles
print(f"\nSpeedup: {speedup:.2f}× (4× BW → {speedup:.1f}× faster)")

---

## 8. MLIR frontend parser (optional)

As a well formed intermediate representation built in the MLIR eco-system, on top of the standard parser from the KTIR-defining project, the KTIR MLIR-frontend, kernels written in KTIR can be parsed with a light-weight parser built out of regex (Python regular expression).
By default `KTIRInterpreter` uses the built-in regex parser.
The MLIR frontend uses the real MLIR C++ bindings for full-fidelity parsing.

See the [README](../README.md#mlir-frontend-bindings-optional) for build and install instructions.

In [ ]:
try:
    from ktir_cpu.mlir_frontend.parser import MLIRFrontendParser
    MLIRFrontendParser()  # raises ImportError if mlir_ktdp not installed
    _mlir_available = True
except ImportError:
    _mlir_available = False
    print("mlir_ktdp not installed — skipping MLIR frontend cells.")
    print("See the install instructions above.")

In [ ]:
if _mlir_available:
    # Pass MLIRFrontendParser directly to the interpreter
    MM, MK, MN = 256, 256, 256
    MBM, MBN, MBK = 128, 128, 128

    mlir_matmul_text = gen_matmul_mlir(MM, MK, MN, MBM, MBN, MBK)

    mlir_interp = KTIRInterpreter(
        latency_config=hw,
        parser=MLIRFrontendParser(),
    )
    mlir_interp.load(mlir_matmul_text)

    mlir_interp.execute_function(
        "matmul_kernel",
        a_ptr=np.random.rand(MM, MK).astype(np.float16),
        b_ptr=np.random.rand(MK, MN).astype(np.float16),
        c_ptr=np.zeros((MM, MN), dtype=np.float16),
        K=MK, BLOCK_SIZE_M=MBM, BLOCK_SIZE_N=MBN, BLOCK_SIZE_K=MBK,
    )

    mlir_report = mlir_interp.get_latency_report()
    print(mlir_report)

In [ ]:
if _mlir_available:
    # Run the same kernel with the regex parser for comparison
    regex_interp = KTIRInterpreter(latency_config=hw)
    regex_interp.load(mlir_matmul_text)
    regex_interp.execute_function(
        "matmul_kernel",
        a_ptr=np.random.rand(MM, MK).astype(np.float16),
        b_ptr=np.random.rand(MK, MN).astype(np.float16),
        c_ptr=np.zeros((MM, MN), dtype=np.float16),
        K=MK, BLOCK_SIZE_M=MBM, BLOCK_SIZE_N=MBN, BLOCK_SIZE_K=MBK,
    )
    regex_report = regex_interp.get_latency_report()

    print(f"regex  cycles={regex_report.kernel_cycles:.0f}  AI={regex_report.core_roofline()['core_AI']:.2f}")
    print(f"mlir   cycles={mlir_report.kernel_cycles:.0f}  AI={mlir_report.core_roofline()['core_AI']:.2f}")
    print("Results are identical — the two parsers produce the same IR.")


## 9. Cross-core communication (comm) view

Every kernel above is embarrassingly parallel, so the `comm` column reads `0`. Communication *is* costed by
the model (`ktdp.inter_tile_reduce` → `comm_cycles`, folded into `total_cycles`; `bottleneck` recognises
`comm`) — it just isn't exercised or surfaced here.

This section loads the in-repo cross-core all-reduce examples (`examples/ktir/ring_reduce*.mlir`) and surfaces
communication: a per-core `compute / memory / comm` breakdown, plus the **critical-path core**'s
`comm_fraction`, `comm_time`, and `comm_bytes`. Metrics are taken on the critical core (`max(total_cycles)`)
— the wall-clock core `kernel_time` comes from — so this is *exposed communication* (PyTorch **HTA** framing):
comm on a non-critical core is hidden behind that core's own work and adds no wall time.

In [ ]:
import pathlib
import matplotlib.pyplot as plt

# Unlike the kernels above (generated inline via gen_*_mlir), the comm examples are
# static .mlir files under examples/ktir/, so resolve the repo root to locate them.
_REPO = pathlib.Path.cwd()
if not (_REPO / "examples" / "ktir").exists():
    _REPO = _REPO.parent   # launched from notebooks/
_KTIR = _REPO / "examples" / "ktir"

def run_comm_kernel(path, func, num_cores=4, n_cols=128, in_ptr=0, out_ptr=512, **run_kwargs):
    """Load an inter-tile reduce example, seed its HBM inputs, run it, return the LatencyReport.

    ``in_ptr`` / ``out_ptr`` are f16 element indices (base_ptr convention); ``hbm.write`` takes
    stick indices, so divide by elements-per-stick (64). ``out_ptr`` must be stick-aligned.
    """
    interp = KTIRInterpreter(latency_config=hw, trace_latency=True)
    interp.load(str(path))
    rng = np.random.default_rng(42)
    rows = rng.uniform(1.0, 2.0, size=(num_cores, n_cols)).astype(np.float16)
    eps = 64  # f16 elements per 128-byte stick
    _orig = interp._prepare_execution
    def _seed(grid_shape):
        _orig(grid_shape)
        interp.memory.hbm.write(in_ptr // eps, rows.flatten())
        interp.memory.hbm.write(out_ptr // eps, np.zeros(n_cols, dtype=np.float16))
    interp._prepare_execution = _seed
    interp.execute_function(func, in_ptr=in_ptr, out_ptr=out_ptr, **run_kwargs)
    return interp.get_latency_report()

def comm_metrics(report):
    """Communication metrics on the critical-path core (exposed comm)."""
    crit = max(report.per_core_summary(), key=lambda r: r["total_cycles"])
    total = crit["total_cycles"]
    breakdown = ({k: crit[f"{k}_cycles"] / total for k in ("compute", "memory", "comm")}
                 if total else {})
    return {
        "critical_core": crit["core_id"],
        "bottleneck": report.bottleneck,
        "comm_fraction": crit["comm_cycles"] / total if total else 0.0,   # exposed comm share
        "comm_time_us": crit["comm_cycles"] / (hw.clock_ghz * 1e3),        # same conversion as kernel_time_us
        "comm_bytes": report.counters[crit["core_id"]].comm_bytes,         # per-core ring bytes
        "temporal_breakdown": breakdown,                                    # 3 ratios, sum = 1
    }

def print_comm_table(report):
    """Critical-core comm metrics + per-core compute/memory/comm table."""
    m = comm_metrics(report)
    print(f"critical core={m['critical_core']}  bottleneck={m['bottleneck']}  "
          f"comm_fraction={m['comm_fraction']*100:.2f}%  "
          f"comm_time={m['comm_time_us']*1e3:.3f} ns  "
          f"comm_bytes={m['comm_bytes']} (per-core)")
    hdr = f"{'core':>4}  {'compute':>10}  {'memory':>10}  {'comm':>10}  {'total':>10}  {'comm%':>6}"
    print(hdr); print("-" * len(hdr))
    for r in report.per_core_summary():
        t = r["total_cycles"]
        print(f"{r['core_id']:>4}  {r['compute_cycles']:>10.3f}  {r['memory_cycles']:>10.3f}  "
              f"{r['comm_cycles']:>10.3f}  {t:>10.3f}  {r['comm_cycles']/t*100:>5.2f}%")

def plot_comm_breakdown(report, title):
    """Per-core stacked bar of compute / memory / comm cycles, with value labels."""
    pcs = report.per_core_summary()
    cores = [r["core_id"] for r in pcs]
    m = comm_metrics(report)
    fig, ax = plt.subplots(figsize=(7, 4))
    bottom = np.zeros(len(cores))
    for name, color in [("compute", "#5B8FF9"), ("memory", "#F6BD16"), ("comm", "#E8684A")]:
        vals = np.array([r[f"{name}_cycles"] for r in pcs])
        ax.bar(cores, vals, bottom=bottom, label=name, color=color, width=0.6, edgecolor="white")
        for x, v, b in zip(cores, vals, bottom):
            if v > 0.05:
                ax.text(x, b + v / 2, f"{v:.2f}", ha="center", va="center",
                        fontsize=8, color="white" if name != "memory" else "#333")
        bottom += vals
    ax.set_xlabel("core"); ax.set_ylabel("cycles"); ax.set_xticks(cores)
    ax.set_title(f"{title}\ncritical core={m['critical_core']} · bottleneck={m['bottleneck']} · "
                 f"comm_fraction={m['comm_fraction']*100:.2f}% · comm_time={m['comm_time_us']*1e3:.3f} ns",
                 fontsize=9)
    ax.legend(frameon=False, ncol=3, loc="upper right")
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    plt.tight_layout(); plt.show()

In [ ]:
# ring_reduce — single-ring all-reduce (one top-level comm)
rr = run_comm_kernel(_KTIR / "ring_reduce.mlir", "ring_reduce")
print_comm_table(rr)
plot_comm_breakdown(rr, "ring_reduce — per-core compute/memory/comm")

In [ ]:
# ring_reduce_inner_loop — same all-reduce inside scf.for (comm scales with iterations)
rril = run_comm_kernel(_KTIR / "ring_reduce_inner_loop.mlir", "ring_reduce_inner_loop", n_iters=3)
print_comm_table(rril)
plot_comm_breakdown(rril, "ring_reduce_inner_loop (n_iters=3) — per-core compute/memory/comm")

---

## Summary

Throughout 9 curated exercises, the notebook demonstrates the usage interface of `ktir-cpu` for verify the correctiness of LLM kernels written in ktir mlir intermediate representation, and for estimating the latency and throughput performances in the framework of roofline models. Below is a complete list of the demonstrated API usages.

```python
from ktir_cpu.interpreter import KTIRInterpreter
from ktir_cpu.latency import HardwareConfig

hw = HardwareConfig()                        # or HardwareConfig(clock_ghz=1.5, ...)
interp = KTIRInterpreter(latency_config=hw)  # add parser=MLIRFrontendParser() for MLIR
interp.load("path/to/kernel.mlir")           # or inline MLIR text

interp.execute_function("kernel_name", arg=tensor, ...)

report = interp.get_latency_report()
print(report)                        # human-readable table + roofline
report.kernel_cycles                 # float
report.kernel_time_us                # float
report.bottleneck                    # 'compute' | 'memory' | 'comm'
report.per_core_summary()            # list[dict] — per-core breakdown
report.chip_roofline()               # dict — AI, compute/dram_throughput, attainment, grid_coverage
report.core_roofline()               # dict — core_AI, core_attainment, units, ...
report.roofline()                    # dict — chip + core merged
```